# Phase 2: FinBERT Baseline on MLflow (Using S3)
This notebook evaluates the off-the-shelf FinBERT model on your test set loaded directly from S3, and logs the results to your live AWS MLflow server.

In [ ]:
!pip install -q mlflow boto3 transformers torch pandas scikit-learn tqdm python-dotenv s3fs

### 1. Load Credentials & Set Up MLflow
We use `.env` or defaults to securely load AWS credentials and MLflow details.

In [ ]:
import os
import mlflow
from dotenv import load_dotenv

# Load environment variables from .env if present
load_dotenv()

AWS_ACCESS_KEY_ID = os.getenv("AWS_ACCESS_KEY_ID")
AWS_SECRET_ACCESS_KEY = os.getenv("AWS_SECRET_ACCESS_KEY")
AWS_DEFAULT_REGION = os.getenv("AWS_DEFAULT_REGION") or "ap-south-1"
S3_BUCKET_NAME = os.getenv("S3_BUCKET_NAME") or "finance-sentiment-mlflow-artifacts-kavishka"
MLFLOW_TRACKING_URI = os.getenv("MLFLOW_TRACKING_URI") or "http://13.235.68.121:5000/"

# Ensure credentials are in the environment for s3fs and boto3 to pick up automatically
os.environ["AWS_ACCESS_KEY_ID"] = AWS_ACCESS_KEY_ID
os.environ["AWS_SECRET_ACCESS_KEY"] = AWS_SECRET_ACCESS_KEY
os.environ["AWS_DEFAULT_REGION"] = AWS_DEFAULT_REGION

# Connect to MLflow
mlflow.set_tracking_uri(MLFLOW_TRACKING_URI)
print("Connected to MLflow at:", mlflow.get_tracking_uri())
print("Using S3 bucket:", S3_BUCKET_NAME)

### 2. Read Test Dataset Directly From S3

In [ ]:
import pandas as pd

s3_path = f"s3://{S3_BUCKET_NAME}/test.csv"
print(f"Reading test dataset from: {s3_path}")

test_df = pd.read_csv(s3_path)
print(f"Loaded {len(test_df)} rows for testing.")
test_df.head()

### 3. Phase 2: FinBERT Zero-Shot Baseline

In [ ]:
import torch
from transformers import pipeline
from sklearn.metrics import accuracy_score, f1_score, classification_report
from tqdm.auto import tqdm
tqdm.pandas()

# Load FinBERT
device = 0 if torch.cuda.is_available() else -1
pipe = pipeline("text-classification", model="ProsusAI/finbert", device=device)

def get_finbert_prediction(text):
    try:
        result = pipe(text, truncation=True, max_length=512)[0]
        return result['label'].lower()
    except Exception as e:
        return "neutral"

print("Running FinBERT on Test Set...")
test_df['finbert_pred'] = test_df['Headline'].progress_apply(get_finbert_prediction)

y_true = test_df['Sentiment'].tolist()
y_pred = test_df['finbert_pred'].tolist()

# Calculate metrics
accuracy = accuracy_score(y_true, y_pred)
f1_macro = f1_score(y_true, y_pred, average='macro')

print(f"\nAccuracy: {accuracy:.4f}")
print(f"F1 (Macro): {f1_macro:.4f}")
print("\nClassification Report:")
print(classification_report(y_true, y_pred))

### 4. Log Baseline to MLflow

In [ ]:
# Create or set the experiment
mlflow.set_experiment("Sentiment_FineTuning_Benchmark")

with mlflow.start_run(run_name="FinBERT_ZeroShot_Baseline"):
    # Log parameters
    mlflow.log_param("model_type", "ProsusAI/finbert")
    mlflow.log_param("evaluation_type", "Zero-Shot")
    mlflow.log_param("test_set_size", len(test_df))
    
    # Log metrics
    mlflow.log_metric("test_accuracy", accuracy)
    mlflow.log_metric("test_f1_macro", f1_macro)
    
    print("✅ Successfully logged metrics to MLflow!")
    print(f"View it live at: {MLFLOW_TRACKING_URI}")